The electric field in a capacitor inspired by Joachim Schöberl

In [ ]:
from netgen.meshing import Mesh as NGMesh, MeshPoint, Element1D, Element0D, Pnt
from ngsolve import *
from netgen.occ import *
from ngsolve.webgui import Draw, FieldLines, AddFieldLines

import matplotlib.pylab as plt

In [ ]:
def CapacitorGeometry():

    air = MoveTo(0, 0).RectangleC(30, 30).Face()
    air.edges.name = "Outer"
    air.faces.name = "air"

    el_u = MoveTo(0, 1).RectangleC(5, 0.5).Face()
    el_u.edges.name = "el_u"
    el_u.faces.name = "el_u"

    el_d = MoveTo(0, -1).RectangleC(5, 0.5).Face()
    el_d.edges.name = "el_d"
    el_d.faces.name = "el_d"

    dielectric = MoveTo(0, 0).RectangleC(4, 1.5).Face()
    dielectric.faces.name = "dielectric"

    shape = Glue([air - dielectric, dielectric])
    shape = shape - el_u - el_d

    shape.edges["el.*"].maxh=0.2
    shape.vertices["el.*"].maxh=0.2
    
    return shape


def CapacitorMesh(shape, h_max):
    
    mesh = Mesh(OCCGeometry(shape, dim=2).GenerateMesh(maxh=h_max))

    return mesh


def CapacitorSolver(mesh, FE_order, epsr):

    fes = H1(mesh, order=FE_order, dirichlet="el.*")

    u = fes.TrialFunction()
    v = fes.TestFunction()

    gfu = GridFunction(fes)
    gfu.Interpolate(mesh.BoundaryCF({"el_u":1, "el_d":-1 }), mesh.Boundaries(".*"))

    a = BilinearForm(epsr*grad(u)*grad(v)*dx).Assemble()
    
    inv = a.mat.Inverse(freedofs=fes.FreeDofs())
    gfu.vec.data -= inv@a.mat * gfu.vec

    return gfu

In [ ]:
geo = CapacitorGeometry()
Draw(geo);

In [ ]:
h_max = 1
mesh = CapacitorMesh(geo, h_max)
Draw (mesh);

In [ ]:
epsr_air = 1.0
epsr_dielectric = 2.0

epsr = mesh.MaterialCF({"air": epsr_air, "dielectric": epsr_dielectric})

Draw(epsr, mesh);

In [ ]:
FE_order = 3

gf_phi = CapacitorSolver(mesh, FE_order, epsr)

In [ ]:
Draw (gf_phi, deformation=True, scale=5);

In [ ]:
fes_flux = HDiv(mesh, order=FE_order-1)

gf_E = GridFunction(fes_flux)
gf_D = GridFunction(fes_flux)
gf_E.Set(-grad(gf_phi))
gf_D.Set(epsr*gf_E)

In [ ]:
Draw (gf_E, mesh, vectors= {"grid_size": 100});

In [ ]:
Draw (Norm(gf_E), mesh, deformation=True, min=0, max=2);

In [ ]:
Draw (gf_D, mesh, vectors= {"grid_size": 100});

In [ ]:
Draw (Norm(gf_D), mesh, deformation=True);

In [ ]:
N = 400
p = [(-10 + 0.05*i, -10 + 0.1*j, 0) for i in range(N) for j in range(N) ] 

fieldlines = gf_E._BuildFieldLines(mesh, p, num_fieldlines=500, length=0.3)

Draw(gf_E, mesh,  "X", draw_vol=True, draw_surf=True, objects=[fieldlines], \
     autoscale=True, min = 0, max = 2, settings={"Objects": {"Surface": False}});

In [ ]:
energy = 0.5 * Integrate(epsr*InnerProduct(gf_E, gf_E), mesh)
print(energy)